# Predictions 1D

<img src="img_notebook/pchip1.png" alt="Descrizione alternativa" width="400" height="300">

#### Librerie

In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from os import listdir
from os.path import isfile, join

from scipy import stats
from scipy.interpolate import CubicSpline, PchipInterpolator, Akima1DInterpolator

import plotly.graph_objects as go

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, DotProduct, Product

import warnings
from sklearn.exceptions import ConvergenceWarning

## Operazioni Preliminari

#### Parametri globali

In [2]:
n_c = 4         # numero di punti curvatura per ogni singolo scenario

#### Caricamento curve AP

In [3]:
# RECUPERO COMBINAZIONI PARAMETRI DoE
data = pd.read_csv('AR_DR_results.csv')

# Definizione dinamica delle feature e dei target
targets = [col for col in data.columns if '(%)' in col]        # I target sono le colonne risultato che contengono '(%)'.
features = [col for col in data.columns if col not in targets] # Le features sono tutte le altre colonne, escludendo i target.

# Stampa di verifica per assicurarsi che le colonne siano state assegnate correttamente
print("Features identificate:", features)
print("Target identificati:", targets)

# Vettori di dati e features
X_data = data[features].to_numpy()
Y_data = data[targets].to_numpy()

# Values
n_run = len(X_data[:,1])
outer_diameter = X_data[:, 0]
wall_thickness = X_data[:, 1]
axial_load = X_data[:, 2]
ext_press = X_data[:, 3]
young_mod = X_data[:, 4]
poiss_mod = X_data[:, 5]
SY1 = X_data[:, 6]
SY2 = X_data[:, 7]
EP2 = X_data[:, 8]
ovalisation = X_data[:, 9]
curvatures = X_data[:, -1]

## LEGGO CURVATURE
def_A = []
for ii in range(1, n_run+1):    
    mypathfile = 'postProc_out/detailed_tables/detailed_run_' + str(ii).zfill(4) + '_results.csv'
    datafrm = pd.read_csv(mypathfile, sep=",", decimal='.')
    dati = datafrm.values
    xs = dati[:,0]  # ascissa curvilinea
    pipe_abs = xs  # nome alternativo 
    def_A.append(dati[:,2])  # area deformata


Features identificate: ['D_out', 'Wall_Thickness', 'Compression_Force', 'External_Pressure', 'Young Modulus', 'SY1', 'SY2', 'EP2', 'Ovalisation', 'Curvature']
Target identificati: ['AR (%)', 'DR (%)']


#### Estrazione dati (AR, DR, param)

In [4]:
# Parametri
step = 100      # punti usati per il grafico
extension = 0.2 # quanta estensione dx e sx del dominio

# Inizializza gli array
num_samples = X_data.shape[0] // n_c   # numero di scenari
params = np.zeros((num_samples, n_c+1))
dr = np.zeros((num_samples, n_c+1))
ar = np.zeros((num_samples, n_c+1))
DR_pred = np.zeros((num_samples, step, 3))
AR_pred = np.zeros((num_samples, step, 3))

# Estrazione di n_c valori per ogni ciclo
for ii in range(num_samples):    
    # Viene aggiunto (0,0) all'inizio in modo automatico
    dr[ii, 1:] = Y_data[ii * n_c:(ii+1) * n_c, -1]  # 4 valori di DR
    ar[ii, 1:] = Y_data[ii * n_c:(ii+1) * n_c, 0]  # 4 valori di AR
    params[ii, 1:] = X_data[ii * n_c:(ii+1) * n_c, -1]  # 4 valori di Curvature

# Verifica
print(f"dimensioni dr = {dr.shape}")
print(f"dimensioni dr = {dr[122]}")

dimensioni dr = (256, 5)
dimensioni dr = [ 0.          0.11776812  1.31750091  7.22414262 21.97339448]


## <span style="color:green"> Single scenario predictions (DR) </span> -> (previsione curva su cui ho 4 punti)

#### Funzione forma e dominio espanso

In [5]:
def compute_shape_and_domain(params, step=100, extension=0.2, mix_par=0.5, mix_mod=0.7):
    """
    Calcola la shape function ibrida e i parametri interpolati (unk_params).
    
    Parameters:
        params (np.array): Vettore dei parametri di curvatura (incluso lo 0 iniziale).
        step (int): Numero di punti interpolati.
        extension (float): Estensione del dominio.
        mix_par (float): Controlla l'estensione della transizione.
        mix_mod (float): Peso della componente PCHIP (1 → solo PCHIP, 0 → solo Poly).

    Returns:
        unk_params (np.array): Dominio esteso su cui interpolare.
        shape_function (np.array): Funzione di peso tra PCHIP e PolyFit.
    """
    params = np.array(params)
    
    min_param, max_param = np.min(params[1:]), np.max(params[1:])
    range_param = max_param - min_param
    ext_range = range_param * (1 + 2 * extension)
    lower_param = max(0, min_param - extension * range_param)

    unk_params = np.linspace(lower_param, lower_param + ext_range, step)

    # Shape function: transizione graduale tra modelli
    shape_function = np.ones(step) * mix_mod
    transition_start = max_param - mix_par * extension * range_param
    transition_end = max_param + mix_par * extension * range_param

    idx1 = np.argmax(unk_params >= transition_start)
    idx2 = np.argmax(unk_params > transition_end)

    if idx2 > idx1:
        shape_function[idx1:idx2] = np.linspace(mix_mod, 1 - mix_mod, idx2 - idx1)
        shape_function[idx2:] = 1 - mix_mod
    else:
        print("Attenzione: transizione shape function mal definita (indici sovrapposti).")

    return unk_params, shape_function


#### Funzione LOO (best hyperparameters) modello ibrido

In [6]:
def loocv_error(params_i, y_i, degree, mix_mod, mix_par, extension=0.2, step=100):
    """
    Calcola l'errore LOOCV escludendo uno dei punti.
    """
    n_points = len(params_i)
    errors = []

    for i in range(1, n_points):
        # Crea set di addestramento escludendo il punto i
        params_train = np.delete(params_i, i)
        y_train = np.delete(y_i, i)
        param_test = params_i[i]
        y_test = y_i[i]

        # Funzione forma ed estensione dominio
        unk_params, shape_function = compute_shape_and_domain(params[i, :], step=100, mix_mod=mix_mod, mix_par=mix_par)
        
        # Interpolazioni
        pchip = PchipInterpolator(params_train, y_train)
        poly_fit = np.polyfit(params_train, y_train, min(degree, len(params_train) - 1))

        y_pchip = pchip(param_test)
        y_poly = np.polyval(poly_fit, param_test)
        y_hybrid = shape_function[np.argmin(np.abs(unk_params - param_test))] * y_pchip + \
                   (1 - shape_function[np.argmin(np.abs(unk_params - param_test))]) * y_poly

        errors.append((y_test - y_hybrid) ** 2)

    return np.mean(errors)

#### Ricerca iperparametri modello ibrido (grid search)

In [7]:
# GRID SEARCH
degrees = [2, 3, 4]
mix_mod_values = [0.65, 0.75, 0,85, 0.9]
mix_par_values = [0.4, 0.55, 0.7, 0.8]

best_config = {}
best_score_dr = float('inf')
best_score_ar = float('inf')

for deg in degrees:
    for mix_mod in mix_mod_values:
        for mix_par in mix_par_values:
            mse_dr, mse_ar = 0.0, 0.0
            for ii in range(num_samples):
                mse_dr += loocv_error(params[ii, :], dr[ii, :], deg, mix_mod, mix_par)
                mse_ar += loocv_error(params[ii, :], ar[ii, :], deg, mix_mod, mix_par)
            mse_dr /= num_samples
            mse_ar /= num_samples
            if mse_dr < best_score_dr:
                best_score_dr = mse_dr
                best_config['dr'] = (deg, mix_mod, mix_par)
            if mse_ar < best_score_ar:
                best_score_ar = mse_ar
                best_config['ar'] = (deg, mix_mod, mix_par)

degree_dr, mix_mod_dr, mix_par_dr = best_config['dr']
degree_ar, mix_mod_ar, mix_par_ar = best_config['ar']

print("Best config DR:", best_config['dr'], "MSE:", best_score_dr)
print("Best config AR:", best_config['ar'], "MSE:", best_score_ar)

Best config DR: (2, 0.65, 0.7) MSE: 31.108106153280268
Best config AR: (2, 0.75, 0.4) MSE: 13.19010092627344


#### Calcolo e stampa modello ibrido con migliori iperparametri

In [8]:
# Estrazione di n_c valori per ogni ciclo
for ii in range(num_samples):
    
    # Viene aggiunto (0,0) all'inizio in modo automatico
    dr[ii, 1:] = Y_data[ii * n_c:(ii+1) * n_c, -1]  # 4 valori di DR
    ar[ii, 1:] = Y_data[ii * n_c:(ii+1) * n_c, 0]  # 4 valori di AR
    params[ii, 1:] = X_data[ii * n_c:(ii+1) * n_c, -1]  # 4 valori di Curvature

    # Funzione forma
    unk_params, shape_function_dr = compute_shape_and_domain(params[ii, :], step=100, mix_mod=mix_mod_dr, mix_par=mix_par_dr)
    unk_params, shape_function_ar = compute_shape_and_domain(params[ii, :], step=100, mix_mod=mix_mod_ar, mix_par=mix_par_ar)
    
    # DR
    dr_interp_pchip = PchipInterpolator(params[ii, :], dr[ii, :])
    predicted_dr = dr_interp_pchip(unk_params)
    DR_pred[ii, :, 0] = predicted_dr    
    
    # Predicition PolyFit
    dr_interp_poly = np.polyfit(params[ii, :], dr[ii, :], degree_dr)
    predicted_dr_poly = np.polyval(dr_interp_poly, unk_params)
    DR_pred[ii, :, 1] = predicted_dr_poly    
    
    # Modello Ibrido 
    DR_pred[ii, :, 2] = shape_function_dr * predicted_dr + (1 - shape_function_dr) * predicted_dr_poly

    # AR
    ar_interp_pchip = PchipInterpolator(params[ii, :], ar[ii, :])
    predicted_ar = ar_interp_pchip(unk_params)
    AR_pred[ii, :, 0] = predicted_ar    
    # Predicition PolyFit
    ar_interp_poly = np.polyfit(params[ii, :], ar[ii, :], degree_ar)
    predicted_ar_poly = np.polyval(ar_interp_poly, unk_params)
    AR_pred[ii, :, 1] = predicted_ar_poly    
    # Modello Ibrido 
    AR_pred[ii, :, 2] = shape_function_ar * predicted_ar + (1 - shape_function_ar) * predicted_ar_poly

#---------------------------------------------------------------------------------------------------------

# PLOTS DR
num_cols = 3   # Numero di colonne
num_rows = (num_samples + num_cols - 1) // num_cols
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows), sharex=True, sharey=True)
axes = axes.flatten()
for kk in range(num_samples):
    ax = axes[kk]
    ax.plot(unk_params, DR_pred[kk, :, 0], label="PCHIP - Model", color='b', linewidth=0.5)
    ax.plot(unk_params, DR_pred[kk, :, 1], label="Poly - Model", color='r', linewidth=0.5)
    ax.plot(unk_params, DR_pred[kk, :, 2], label="Hybrid - Model", color='g', linewidth=2)
    ax.scatter(params[kk,1:], dr[kk,1:], color='black')  
    ax.scatter(params[kk,0], dr[kk,0], color='grey') 
    ax.set_title(f"Set {kk}, wt={wall_thickness[kk*n_c]}  D={outer_diameter[kk*n_c]}  C={axial_load[kk*n_c]}", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(True)
# Rimuove subplot vuoti se `num_samples` non è multiplo di `num_cols`
for kk in range(num_samples, len(axes)):
    fig.delaxes(axes[kk])
fig.suptitle(f"DR  mix_mod={mix_mod_dr}, mix_par={mix_par_dr}, deg={degree_dr}", fontsize=20, y=1)
fig.supxlabel("Curvature", fontsize=12)
fig.supylabel("DR", fontsize=12)
fig.subplots_adjust(hspace=0.3, wspace=0.3) 
plt.tight_layout()
plt.show()

#---------------------------------------------------------------------------------------------------------

# PLOTS AR
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows), sharex=True, sharey=True)
axes = axes.flatten()
for kk in range(num_samples):
    ax = axes[kk]
    ax.plot(unk_params, AR_pred[kk, :, 0], label="PCHIP - Model", color='b', linewidth=0.5)
    ax.plot(unk_params, AR_pred[kk, :, 1], label="Poly - Model", color='r', linewidth=0.5)
    ax.plot(unk_params, AR_pred[kk, :, 2], label="Hybrid - Model", color='g', linewidth=2)
    ax.scatter(params[kk,1:], ar[kk,1:], color='black')  
    ax.scatter(params[kk,0], ar[kk,0], color='grey') 
    ax.set_title(f"Set {kk}, wt={wall_thickness[kk*n_c]}  D={outer_diameter[kk*n_c]}  C={axial_load[kk*n_c]}", fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(True)
# Rimuove subplot vuoti se `num_samples` non è multiplo di `num_cols`
for kk in range(num_samples, len(axes)):
    fig.delaxes(axes[kk])
fig.suptitle(f"AR  mix_mod={mix_mod_ar}, mix_par={mix_par_ar}, deg={degree_ar}", fontsize=20, y=1)
fig.supxlabel("Curvature", fontsize=12)
fig.supylabel("AR", fontsize=12)
fig.subplots_adjust(hspace=0.3, wspace=0.3) 
plt.tight_layout()
plt.show()